# 1 — Merge Datasets
**Course:** Household Economics  
**Author:** Koray Aktas, Giacomo Faccin, Ca' Foscari Venezia  
**Data:** SHIW 1990–2022 — Bank of Italy

---
This notebook merges all raw SHIW modules into a single dataset.

**Key identifiers:**
- `nquest` — household questionnaire ID (primary key at household level)
- `nord` — individual member ID within the household
- `anno` — survey year
- Together `nquest + nord + anno` uniquely identify an individual observation
- `nordp` links to the same individual in the **previous wave** (panel component)

**Modules merged:**
| File | Level | Content |
|------|-------|---------|
| `comp` | Individual | Demographics, employment status |
| `cons` | Household | Consumption, income, savings |
| `rfam` | Household | Income components (labour, pension, self-empl, property) |
| `ricf` | Household | Net wealth, financial/real assets & liabilities |
| `peso` | Household | Survey weights |
| `ldip` | Individual | Dependent employment details |
| `defl` | Year | Deflator / revaluation coefficients |

## 1. Setup & Load Config

In [9]:
import os, json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Option A: load paths from config.json created by notebook 0 ──────────────
#config_path = "/Volumes/TOSHIBA/HHEc/config.json"
#with open(config_path) as f:
#     PATHS = json.load(f)

# ── Option B: set paths manually (if you skipped notebook 0) ─────────────────

BASE_DIR = "C:/Users/Utente/OneDrive/Immagini/Documenti/Economic_Risk_work"   # ← change this
PATHS = {
    "DataIn":  os.path.join(BASE_DIR, "DataIn"),
    "DataOut": os.path.join(BASE_DIR, "DataOut"),
}

DataIn  = PATHS["DataIn"]
DataOut = PATHS["DataOut"]
os.makedirs(DataOut, exist_ok=True)

print(f"DataIn  → {DataIn}")
print(f"DataOut → {DataOut}")

DataIn  → C:/Users/Utente/OneDrive/Immagini/Documenti/Economic_Risk_work\DataIn
DataOut → C:/Users/Utente/OneDrive/Immagini/Documenti/Economic_Risk_work\DataOut


## 2. Load Base Module: `comp` (Individual Demographics)

In [10]:
comp = pd.read_stata(os.path.join(DataIn, "comp.dta"))
print(f"comp:  {comp.shape[0]:,} rows × {comp.shape[1]} cols")
comp.head(3)

comp:  477,387 rows × 43 cols


,nquest,nord,anno,ireg,iprov,acom5,par,sesso,eta,eta5,...,perc,perl,nperc,nperl,ncomp,enasc2,etapen,asnonoc,qualp10n,asnonoc2
0,19770001,1,1977,1,1.0,5,1.0,1,70.0,5,...,1,0,1,0,2,NaN,NaN,NaN,NaN,NaN
1,19770001,2,1977,1,1.0,5,2.0,2,58.5,4,...,0,0,1,0,2,NaN,NaN,NaN,NaN,NaN
2,19770002,1,1977,1,1.0,5,1.0,2,58.5,4,...,1,0,2,1,2,NaN,NaN,NaN,NaN,NaN


## 3. Merge Household-Level Modules (m:1 on nquest + anno)

In [11]:
# Helper: merge and report match statistics (equivalent to Stata's 'tab _merge')
def merge_report(left, right_path, on, how="left", suffixes=("_x", "_y")):
    right = pd.read_stata(right_path)
    fname = os.path.basename(right_path)
    # drop duplicate columns that already exist in left (except key cols)
    key_cols = on if isinstance(on, list) else [on]
    dup_cols = [c for c in right.columns if c in left.columns and c not in key_cols]
    right = right.drop(columns=dup_cols, errors='ignore')
    merged = left.merge(right, on=on, how=how, indicator=True)
    counts = merged['_merge'].value_counts()
    print(f"\nMerge with {fname} (on {on}):")
    print(counts.to_string())
    merged = merged.drop(columns='_merge')
    return merged

# Household-level merges (m:1 → left merge on nquest + anno)
df = merge_report(comp, os.path.join(DataIn, "cons.dta"), on=["nquest", "anno"])
df = merge_report(df,   os.path.join(DataIn, "rfam.dta"), on=["nquest", "anno"])
df = merge_report(df,   os.path.join(DataIn, "ricf.dta"), on=["nquest", "anno"])
df = merge_report(df,   os.path.join(DataIn, "peso.dta"), on=["nquest", "anno"])

print(f"\nAfter household merges: {df.shape[0]:,} rows × {df.shape[1]} cols")


Merge with cons.dta (on ['nquest', 'anno']):
_merge
both          477387
left_only          0
right_only         0

Merge with rfam.dta (on ['nquest', 'anno']):
_merge
both          477387
left_only          0
right_only         0

Merge with ricf.dta (on ['nquest', 'anno']):
_merge
both          477387
left_only          0
right_only         0

Merge with peso.dta (on ['nquest', 'anno']):
_merge
both          477387
left_only          0
right_only         0

After household merges: 477,387 rows × 97 cols


## 4. Merge Individual-Level Modules (1:m on nquest + nord + anno)

In [12]:
# Individual-level merges
df = merge_report(df, os.path.join(DataIn, "ldip.dta"), on=["nquest", "nord", "anno"])

# rper.dta (individual wages) — include if available
rper_path = os.path.join(DataIn, "rper.dta")
if os.path.exists(rper_path):
    df = merge_report(df, rper_path, on=["nquest", "nord", "anno"])
else:
    print("\nrper.dta not found — skipping individual wage merge.")

print(f"\nAfter individual merges: {df.shape[0]:,} rows × {df.shape[1]} cols")


Merge with ldip.dta (on ['nquest', 'nord', 'anno']):
_merge
left_only     340940
both          137605
right_only         0

Merge with rper.dta (on ['nquest', 'nord', 'anno']):
_merge
both          296888
left_only     181657
right_only         0

After individual merges: 478,545 rows × 108 cols


## 5. Select Variables of Interest

In [13]:
# Equivalent to Stata's 'keep' command
KEEP_VARS = [
    # Identifiers
    "nquest", "nord", "anno",
    # Demographics
    "cfdic", "sesso", "nordp", "par", "enasc", "nperc", "nascarea", "nascreg",
    "ncomp", "anasc", "staciv", "studio", "qualp7n", "settp7", "nonoc", "eta", "area3",
    # Income & consumption
    "y1", "c", "cn", "cd", "cd1", "cd2", "s1",
    "yl", "yt", "ym", "yc",
    # Wealth
    "w", "ar", "ar1", "ar2", "ar3", "af", "pf",
    # Survey weights
    "peso", "pesopop",
    # Employment details
    "partime", "mesilav", "contratt", "dimaz", "ylm", "oretot", "orestra",
    # Education extras
    "yl1",
]

# Optional columns that may or may not be present
OPTIONAL_VARS = ["consal", "condiv", "stupcf", "stumcf", "enasc"]

existing = [v for v in KEEP_VARS if v in df.columns]
optional_existing = [v for v in OPTIONAL_VARS if v in df.columns]
all_keep = list(dict.fromkeys(existing + optional_existing))  # preserve order, no dupes

missing = [v for v in KEEP_VARS if v not in df.columns]
if missing:
    print(f"Variables not found (may be in rper/fami): {missing}")

df = df[all_keep].copy()
print(f"\nDataset after variable selection: {df.shape[0]:,} rows × {df.shape[1]} cols")

Variables not found (may be in rper/fami): ['enasc']

Dataset after variable selection: 478,545 rows × 47 cols


## 6. Merge Deflator & Filter Years

In [14]:
defl = pd.read_stata(os.path.join(DataIn, "defl.dta"))[["anno", "rival"]]

df = df.merge(defl, on="anno", how="inner")  # keepif _merge==3 → inner
print(f"After deflator merge: {df.shape[0]:,} rows")

# Keep only 1990 onwards
df = df[df["anno"] >= 1990].copy()
print(f"After filtering anno >= 1990: {df.shape[0]:,} rows")

print("\nYears present:")
print(df["anno"].value_counts().sort_index().to_string())

After deflator merge: 478,545 rows
After filtering anno >= 1990: 311,903 rows

Years present:
anno
1991    24930
1993    24050
1995    23955
1998    20937
2000    22382
2002    21195
2004    20622
2006    19621
2008    19961
2010    19885
2012    20076
2014    19400
2016    16508
2020    15248
2022    23133


## 7. Sort & Save

In [15]:
df = df.sort_values(["nquest", "nord", "anno"]).reset_index(drop=True)

out_path = os.path.join(DataOut, "data_SHIW2022.pkl")
df.to_pickle(out_path)

# Also save as CSV for interoperability
df.to_csv(os.path.join(DataOut, "data_SHIW2022.csv"), index=False)

print(f"Saved to: {out_path}")
print(f"Final dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nNotebook 1 — Merge complete ✓")
df.head()

Saved to: C:/Users/Utente/OneDrive/Immagini/Documenti/Economic_Risk_work\DataOut\data_SHIW2022.pkl
Final dataset: 311,903 rows × 48 columns

Notebook 1 — Merge complete ✓


,nquest,nord,anno,cfdic,sesso,nordp,par,nperc,nascarea,nascreg,...,pesopop,partime,mesilav,contratt,dimaz,ylm,oretot,orestra,yl1,rival
0,25,1,1991,1,1,1.0,1.0,2,3.0,16.0,...,8223.587891,0.0,12.0,NaN,NaN,14460.792969,45.0,6.0,25306.388672,2.115668
1,25,1,1993,1,1,1.0,1.0,2,3.0,16.0,...,3342.373779,0.0,12.0,NaN,7.0,16526.621094,39.0,6.0,27888.671875,1.908828
2,25,1,1995,1,1,1.0,1.0,2,3.0,16.0,...,4761.857422,0.0,12.0,NaN,7.0,23240.560547,45.0,6.0,36151.984375,1.700596
3,25,2,1991,0,2,2.0,2.0,2,3.0,16.0,...,8223.587891,0.0,12.0,NaN,NaN,10845.594727,21.0,0.0,25306.388672,2.115668
4,25,2,1993,0,2,2.0,2.0,2,3.0,16.0,...,3342.373779,0.0,12.0,NaN,7.0,11362.051758,18.0,0.0,27888.671875,1.908828
